# Volatility surfaces and Dupire local volatility

A walkthrough of the `market_structures/volatility/` and `montecarlo/volatility/` packages, organised around one question: **what does it cost to assume Black-Scholes when the market actually quotes a skew?**

The notebook is structured as a transition from the simplest model (constant $\sigma$) to the full Dupire local-volatility bridge:

1. **The Black-Scholes assumption.** A single scalar $\sigma$, what it implies, and how the market disagrees.
2. **Implied-volatility surfaces.** `InterpolatedVolSurface` non-parametric, then `SVISurface` and `SSVISurface` parametric.
3. **Arbitrage diagnostics.** Calendar and butterfly checks (`durrleman_g`, `check_calendar`, `check_butterfly`).
4. **From quote to diffusion.** The `VolModel` hierarchy: `ConstantVol` → `BlackTermStructureVol` → `DupireLocalVol`.
5. **Pricing impact.** Reprice a strip of European calls via a small Euler-log Monte Carlo under each model and show where the constant-vol assumption breaks.

## Environment setup

The cell below calls `setup_demo_env()` from `examples/_setup.py`, which performs three steps:

1. Adds the project root to `sys.path` so first-party packages (`database`, `schedules`, `market_structures`, `credit`, ...) import without installation.
2. Redirects the global SQLite path to `examples/demo.db` via `set_db_path()`. The production `quant.db` is never read or written by this notebook.
3. Creates the domain tables and seeds the holidays table (USD/EUR/GBP/PLN, years 2000-2100) on first run; subsequent runs detect existing data and skip seeding.

A status line is printed when the cell runs so you can see which path was taken.

In [ ]:
import sys
sys.path.insert(0, "..")

from examples._setup import setup_demo_env
setup_demo_env()

from datetime import date

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

from market_structures.volatility import (
    InterpolatedVolSurface,
    PowerLawPhi,
    SSVISurface,
    SVIParameters,
    SVISlice,
    SVISurface,
    black_scholes_price,
    check_butterfly,
    check_calendar,
    durrleman_g,
    fit_ssvi,
    fit_svi_slice,
    fit_svi_surface,
)
from montecarlo import (
    BlackTermStructureVol,
    ConstantVol,
    DupireLocalVol,
    MersenneTwisterSampler,
    WichuraAS241Transform,
    make_normal_sampler,
)

REF_DATE = date(2026, 1, 1)
SPOT = 100.0
FORWARD = lambda T: SPOT  # zero rates and dividends for cleanest demo

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. The Black-Scholes assumption

Black-Scholes assumes spot follows a geometric Brownian motion with a **single, constant** volatility $\sigma$:

$$
dS_t = (r - q)\,S_t\,dt + \sigma\,S_t\,dW_t.
$$

The market never quotes $\sigma$. It quotes **call prices** $C(T, K)$, which we invert to *implied volatility* $\sigma_\text{imp}(T, K)$ via Black-Scholes:

$$
C_\text{market}(T, K) = C_\text{BS}\!\bigl(S_0,\,K,\,T,\,\sigma_\text{imp}(T, K)\bigr).
$$

If Black-Scholes were a true description of the world, $\sigma_\text{imp}(T, K)$ would be a constant. In equities it is anything but: the function curves down in $K$ (negative skew), curves up in the wings (smile), and grows in $T$ (term structure).

The cell below builds a synthetic but realistic equity surface (3 expiries × 5 strikes, monotone negative skew, mild term-structure) and inverts it through the project's pipeline.

In [ ]:
expiries = [0.5, 1.0, 2.0]
strikes = np.array([80.0, 90.0, 100.0, 110.0, 120.0])
# Equity-style skew: lower-strike vols higher than upper-strike, term-structure rising.
implied_vols_market = np.array([
    [0.27, 0.24, 0.22, 0.20, 0.21],  # T = 0.5
    [0.25, 0.22, 0.20, 0.19, 0.20],  # T = 1.0
    [0.23, 0.21, 0.19, 0.18, 0.19],  # T = 2.0
])

log_moneynesses = [float(np.log(K / SPOT)) for K in strikes]
total_variances = [
    [implied_vols_market[i, j] ** 2 * expiries[i] for j in range(len(strikes))]
    for i in range(len(expiries))
]

surface_market = InterpolatedVolSurface(
    reference_date=REF_DATE,
    forward=FORWARD,
    expiries=expiries,
    log_moneynesses=[log_moneynesses] * len(expiries),
    total_variances=total_variances,
)

fig, ax = plt.subplots()
for i, T in enumerate(expiries):
    ax.plot(strikes, implied_vols_market[i], "-o", label=f"T = {T:.2f}")
sigma_atm_avg = implied_vols_market[:, len(strikes) // 2].mean()
ax.axhline(sigma_atm_avg, color="grey", linestyle="--", alpha=0.5,
           label=f"Average ATM σ = {sigma_atm_avg:.3f} (BS would assume this)")
ax.set_xlabel("strike K")
ax.set_ylabel("implied volatility")
ax.set_title("Quoted implied vols vs Black-Scholes constant-σ assumption")
ax.legend()
plt.show()

## 2. Parametric surfaces: SVI and SSVI

`InterpolatedVolSurface` is non-parametric — it stores the implied vols you hand it and interpolates linearly in total variance $w(T, k_\text{log}) = \sigma_\text{imp}^2 T$. That is great for fidelity at the quotes but gives you no analytical structure off-pillar.

**Raw SVI** (Gatheral 2004) is a five-parameter slice model:

$$
w(k) = a + b\Bigl(\rho (k - m) + \sqrt{(k - m)^2 + \sigma^2}\Bigr),
\qquad
b \geq 0,\;\;|\rho| \leq 1,\;\;\sigma > 0.
$$

The parameters split cleanly: $a$ sets the floor, $b$ the wing slope, $\rho$ the skew, $m$ the vertex location, $\sigma$ the vertex smoothness. The project ships a quasi-explicit calibrator (De Marco–Martini 2009 / Zeliade Systems): for fixed $(m, \sigma)$ the inner problem is a constrained linear least-squares in $(a, b\rho, b)$; the outer 2-D Nelder–Mead search escapes the local optima that a direct 5-D fit lands in.

In [ ]:
# Fit SVI to each slice in turn.
surface_svi = fit_svi_surface(
    reference_date=REF_DATE,
    forward=FORWARD,
    expiries=expiries,
    log_moneynesses_by_slice=[log_moneynesses] * len(expiries),
    total_variances_by_slice=total_variances,
)

# Plot the calibrated slices vs the market dots.
k_plot = np.linspace(min(log_moneynesses) - 0.05, max(log_moneynesses) + 0.05, 200)
fig, axes = plt.subplots(1, len(expiries), figsize=(15, 4), sharey=True)
for i, (T, slc, ax) in enumerate(zip(expiries, surface_svi.slices, axes)):
    sigma_curve = np.sqrt(np.array([slc.total_variance(float(k)) for k in k_plot]) / T)
    ax.plot(k_plot, sigma_curve, label="SVI fit", color="tab:blue")
    ax.plot(log_moneynesses, implied_vols_market[i], "o", color="tab:red", label="market")
    ax.set_title(f"T = {T:.2f}\n a={slc.params.a:.4f} b={slc.params.b:.3f} ρ={slc.params.rho:+.3f}\n"
                 f" m={slc.params.m:+.4f} σ={slc.params.sigma:.3f}")
    ax.set_xlabel("k_log = log(K/F)")
    if i == 0:
        ax.set_ylabel("implied σ")
    ax.legend()
plt.suptitle("Quasi-explicit SVI fit — one slice per expiry")
plt.tight_layout()
plt.show()

**SSVI** (Gatheral–Jacquier 2014) is a *global* parameterisation tying every slice together through an ATM term structure $\theta_T = w(T, 0)$ and a phi function $\phi(\theta)$:

$$
w(T, k) = \tfrac{\theta_T}{2}\Bigl(1 + \rho\,\phi(\theta_T)\,k + \sqrt{(\phi(\theta_T) k + \rho)^2 + 1 - \rho^2}\Bigr).
$$

The ATM identity $w(T, 0) = \theta_T$ is automatic. The project ships two phi families — power-law $\phi(\theta) = \eta\,\theta^{-\gamma}(1 + \theta)^{\gamma - 1}$ and Heston-like $\phi(\theta) = \frac{1}{\lambda\theta}\bigl(1 - \frac{1 - e^{-\lambda\theta}}{\lambda\theta}\bigr)$ — fit jointly with $\rho$ by Levenberg–Marquardt.

In [ ]:
surface_ssvi = fit_ssvi(
    reference_date=REF_DATE,
    forward=FORWARD,
    expiries=expiries,
    log_moneynesses_by_slice=[log_moneynesses] * len(expiries),
    total_variances_by_slice=total_variances,
    phi_kind="power_law",
)

print("SSVI fit:")
print(f"  rho            = {surface_ssvi.rho:+.4f}")
print(f"  phi family     = {surface_ssvi.phi.kind}")
print(f"  phi params     = {tuple(round(p, 4) for p in surface_ssvi.phi.params)}")
print(f"  theta_T pillars= {[round(t, 5) for t in surface_ssvi.theta_atm]}")

# Side-by-side: SVI vs SSVI at the middle expiry.
T_mid = expiries[1]
sigma_svi = np.sqrt(np.array([surface_svi.total_variance(T_mid, float(k)) for k in k_plot]) / T_mid)
sigma_ssvi = np.sqrt(np.array([surface_ssvi.total_variance(T_mid, float(k)) for k in k_plot]) / T_mid)
fig, ax = plt.subplots()
ax.plot(k_plot, sigma_svi, label="SVI (per-slice)")
ax.plot(k_plot, sigma_ssvi, label="SSVI (global)", linestyle="--")
ax.plot(log_moneynesses, implied_vols_market[1], "o", color="tab:red", label="market")
ax.set_xlabel("k_log = log(K/F)")
ax.set_ylabel("implied σ")
ax.set_title(f"Parametric fits at T = {T_mid:.2f}: SVI (5 params) vs SSVI (3 globals + θ-pillars)")
ax.legend()
plt.show()

## 3. Arbitrage diagnostics

Two no-arbitrage conditions must hold across a surface:

- **Calendar**: $\partial w / \partial T \geq 0$ everywhere — vanilla calls at a longer expiry are at least as expensive as the same-strike call at a shorter expiry.
- **Butterfly** (Roper 2010 / Gatheral–Jacquier 2014): the *Durrleman density function*

$$
g(k) = \Bigl(1 - \frac{k\,w'(k)}{2 w(k)}\Bigr)^2 - \frac{w'(k)^2}{4}\Bigl(\frac{1}{w(k)} + \frac{1}{4}\Bigr) + \frac{w''(k)}{2}
$$

must be non-negative; negative $g$ corresponds to a negative state-price density at $K = F\,e^k$, i.e. an arbitrageable wing.

The project's `arbitrage.py` module exposes `durrleman_g`, `check_butterfly`, and `check_calendar` on any `DifferentiableVolSurface` (SVI / SSVI). On a plain `VolSurface` the derivatives are not closed-form, so the checks are not available there — fit a parametric surface first.

In [ ]:
k_check = np.linspace(-1.5, 1.5, 401)
T_check = np.linspace(0.25, 2.5, 30)

# Calendar check over the SSVI surface.
cal_report = check_calendar(surface_ssvi, T_check, k_check)
print(f"Calendar check (SSVI): arb-free = {cal_report.is_arb_free},"
      f" min ∂w/∂T = {cal_report.min_dw_dT:+.3e}"
      f" at (T={cal_report.T_at_min:.3f}, k={cal_report.k_at_min:+.3f})")

# Butterfly check at each expiry slice.
fig, ax = plt.subplots()
for T in expiries:
    g_curve = np.array([durrleman_g(surface_ssvi, T, float(k)) for k in k_check])
    ax.plot(k_check, g_curve, label=f"T = {T:.2f}")
    butt = check_butterfly(surface_ssvi, T, k_check)
    marker = "✓" if butt.is_arb_free else "✗"
    print(f"  Butterfly at T={T:.2f}: arb-free = {butt.is_arb_free} {marker}"
          f"  min g = {butt.g_min:+.3e} at k={butt.k_at_min:+.3f}")
ax.axhline(0.0, color="grey", linestyle="--", alpha=0.5)
ax.set_xlabel("k_log")
ax.set_ylabel("Durrleman g(k)")
ax.set_title("Butterfly density: g(k) must be non-negative everywhere")
ax.legend()
plt.show()

## 4. From quotes to diffusion: the `VolModel` hierarchy

A `VolSurface` is a **quote object**: it gives you implied vols by $(T, K)$. It is *not* a diffusion coefficient — you cannot feed it directly to a path engine, because a path engine evolves spot in *real* time and needs to know the local instantaneous variance at $(t, S_t)$.

The `montecarlo.volatility` package provides three diffusion-side models on a common ABC:

```python
class VolModel:
    def diffusion(time, spot, state=None) -> np.ndarray:
        """Per-path instantaneous sigma at (time, spot)."""
```

| Model | What it reads from the surface | What it reprices |
|---|---|---|
| `ConstantVol(σ)` | Nothing — a single scalar. | Only the one strike/expiry whose implied vol equals σ. |
| `BlackTermStructureVol(surface)` | The ATM column $\sigma_\text{imp}(T, F(T))$. Builds a piecewise-constant $\sigma_\text{inst}(t)$ such that $\int_0^{T_i} \sigma_\text{inst}^2\,du = \sigma_\text{imp}(T_i, F)^2 T_i$. | Every ATM vanilla at each pillar expiry, exactly. Skew is ignored. |
| `DupireLocalVol(surface)` | The full $w(T, k)$. Applies Dupire's formula in canonical coordinates. | **Every** vanilla — ATM and skew, at every $(T, K)$ — by construction. |

Dupire's formula in $(T, k_\text{log}, w)$ coordinates is:

$$
\sigma_\text{loc}^2(T, k_\text{log}) = \frac{\partial w / \partial T}{1 - \dfrac{k_\text{log}}{2 w} \dfrac{\partial w}{\partial k_\text{log}} + \tfrac{1}{4}\Bigl(-\tfrac{1}{4} - \tfrac{1}{w} + \tfrac{k_\text{log}^2}{w^2}\Bigr)\Bigl(\dfrac{\partial w}{\partial k_\text{log}}\Bigr)^2 + \tfrac{1}{2} \dfrac{\partial^2 w}{\partial k_\text{log}^2}}.
$$

For a `DifferentiableVolSurface` (SVI, SSVI), all four canonical quantities are closed-form, so Dupire is evaluated analytically. For a plain `VolSurface` we pre-compute $\sigma_\text{loc}^2$ on a 50 × 100 $(t, k_\text{log})$ grid with central finite differences and bilinearly interpolate at runtime.

In [ ]:
# Build all three models on the SSVI surface so we can plot σ_loc vs σ_imp side by side.
model_const = ConstantVol(sigma=sigma_atm_avg)
model_term = BlackTermStructureVol(surface_ssvi)
model_dupire = DupireLocalVol(surface_ssvi)

# Sample σ(t, S) on a grid for the heatmaps.
t_axis = np.linspace(0.1, 2.0, 80)
S_axis = np.linspace(70.0, 130.0, 80)
T_mesh, S_mesh = np.meshgrid(t_axis, S_axis, indexing="ij")

# σ_imp(T, K) read directly from the SSVI surface — for comparison.
sigma_imp_field = np.empty_like(T_mesh)
sigma_loc_field = np.empty_like(T_mesh)
sigma_term_field = np.empty_like(T_mesh)
for i, T in enumerate(t_axis):
    for j, S in enumerate(S_axis):
        sigma_imp_field[i, j] = surface_ssvi.implied_vol(T, float(S))
        sigma_loc_field[i, j] = float(model_dupire.diffusion(T, np.float64(S)))
        sigma_term_field[i, j] = float(model_term.diffusion(T, np.float64(S)))

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
vmin = min(sigma_imp_field.min(), sigma_loc_field.min(), sigma_term_field.min())
vmax = max(sigma_imp_field.max(), sigma_loc_field.max(), sigma_term_field.max())
for ax, field, title in zip(
    axes,
    [sigma_imp_field, sigma_term_field, sigma_loc_field],
    [
        "Implied σ(T, K) — the surface itself",
        "BlackTermStructureVol σ(t) — flat in S, ATM-only",
        "Dupire σ_loc(t, S) — skew-aware diffusion",
    ],
):
    pcm = ax.pcolormesh(S_axis, t_axis, field, vmin=vmin, vmax=vmax, cmap="viridis", shading="auto")
    ax.set_xlabel("S")
    ax.set_title(title)
axes[0].set_ylabel("t (years)")
fig.colorbar(pcm, ax=axes, label="σ")
plt.show()

Two things are visible in the heatmaps:

- The middle panel (`BlackTermStructureVol`) has no $S$-dependence — the column is constant in $S$ at every $t$. The model knows the term structure but ignores the skew entirely.
- The right panel (`DupireLocalVol`) bends in $S$ in the same direction the skew bends in $K$ — but it's a *steeper* bend, because the local-vol skew is roughly twice the implied-vol skew at short expiry (Gatheral's "rule of two" / Derman–Kani sticky-strike rule). Read the panels side-by-side: implied vol at $K = 80$ might be 0.27, but the *local* vol at $S = 80$ is meaningfully higher than that.

This is exactly why Dupire is the right diffusion when you want a single Markov SDE in $S_t$ alone that reprices the full vanilla strip.

## 5. Pricing impact

Time to put a number on it. We reprice a strip of European calls at expiry $T = 1.0$ under each of the three models using a small Euler-log Monte Carlo:

$$
\log S_{t + \Delta t} = \log S_t - \tfrac{1}{2}\,\sigma(t, S_t)^2 \Delta t + \sigma(t, S_t)\,\sqrt{\Delta t}\,Z, \quad Z \sim \mathcal{N}(0, 1).
$$

The path engine is intentionally minimal — variance reduction, Brownian-bridge dimension allocation, and proper antithetics belong in the path-engine PR. Here we use the project's `MersenneTwisterSampler` + `WichuraAS241Transform` pair (machine-precision tails) with 50 000 paths × 200 steps to get standard errors of order 1% on the option prices, enough to distinguish the three models cleanly.

The *baseline* is the surface itself: for each strike $K$ we compute the Black-Scholes price at the market-quoted implied vol $\sigma_\text{imp}(T, K)$. Any MC price that disagrees with this baseline is a model that fails to reprice the input.

In [ ]:
def euler_log_mc(
    model,
    *,
    S0: float,
    T: float,
    strikes: np.ndarray,
    n_paths: int = 50_000,
    n_steps: int = 200,
    seed: int = 42,
):
    """Tiny Euler-log MC path engine, just enough to price a strip of vanilla calls.

    Returns (price_per_strike, stderr_per_strike) under zero rates and dividends.
    """
    dt = T / n_steps
    sqrt_dt = np.sqrt(dt)
    nsamp = make_normal_sampler(
        MersenneTwisterSampler(seed=seed),
        WichuraAS241Transform(),
    )
    Z_block = nsamp.next_block(n_paths, n_steps)  # shape (n_paths, n_steps)
    log_S = np.full(n_paths, np.log(S0), dtype=np.float64)
    t = 0.0
    for step in range(n_steps):
        # Evaluate at the START of the step (Euler convention).
        t_eval = t if step > 0 else 1e-8  # σ_loc requires t > 0
        sigma = model.diffusion(t_eval, np.exp(log_S))
        log_S += -0.5 * sigma * sigma * dt + sigma * sqrt_dt * Z_block[:, step]
        t += dt
    S_T = np.exp(log_S)
    prices = np.empty(strikes.size)
    stderrs = np.empty(strikes.size)
    for j, K in enumerate(strikes):
        payoff = np.maximum(S_T - K, 0.0)
        prices[j] = float(payoff.mean())
        stderrs[j] = float(payoff.std(ddof=1) / np.sqrt(n_paths))
    return prices, stderrs


T_pricing = 1.0
strike_grid = np.linspace(80.0, 120.0, 9)

# Baseline: BS prices at the market implied vols (exact target by definition).
baseline = np.array([
    black_scholes_price(
        forward=SPOT, strike=float(K), time_to_expiry=T_pricing,
        sigma=surface_ssvi.implied_vol(T_pricing, float(K)),
        df_funding=1.0, option_type="C",
    )
    for K in strike_grid
])

prices_const, se_const = euler_log_mc(model_const, S0=SPOT, T=T_pricing, strikes=strike_grid)
prices_term, se_term = euler_log_mc(model_term, S0=SPOT, T=T_pricing, strikes=strike_grid)
prices_dup, se_dup = euler_log_mc(model_dupire, S0=SPOT, T=T_pricing, strikes=strike_grid)

In [ ]:
# Express the mis-pricing as the implied-vol error so the y-axis is interpretable.
from market_structures.volatility import implied_vol_from_price


def invert_iv(prices, K_grid, T):
    out = np.empty_like(prices)
    for j, K in enumerate(K_grid):
        try:
            out[j] = implied_vol_from_price(
                price=float(prices[j]), forward=SPOT, strike=float(K),
                time_to_expiry=T, df_funding=1.0, option_type="C",
            )
        except Exception:
            out[j] = np.nan
    return out


iv_market = np.array([surface_ssvi.implied_vol(T_pricing, float(K)) for K in strike_grid])
iv_const = invert_iv(prices_const, strike_grid, T_pricing)
iv_term = invert_iv(prices_term, strike_grid, T_pricing)
iv_dup = invert_iv(prices_dup, strike_grid, T_pricing)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
ax1.plot(strike_grid, iv_market, "k-", linewidth=2, label="market σ_imp (target)")
ax1.plot(strike_grid, iv_const, "o-", label=f"ConstantVol(σ={sigma_atm_avg:.3f})")
ax1.plot(strike_grid, iv_term, "s-", label="BlackTermStructureVol")
ax1.plot(strike_grid, iv_dup, "^-", label="DupireLocalVol")
ax1.set_xlabel("strike K")
ax1.set_ylabel("implied σ inverted from MC price")
ax1.set_title(f"σ_imp(T={T_pricing}, K) under each diffusion model")
ax1.legend()

ax2.axhline(0.0, color="k", linewidth=1)
ax2.plot(strike_grid, (iv_const - iv_market) * 100, "o-", label="ConstantVol")
ax2.plot(strike_grid, (iv_term - iv_market) * 100, "s-", label="BlackTermStructureVol")
ax2.plot(strike_grid, (iv_dup - iv_market) * 100, "^-", label="DupireLocalVol")
ax2.set_xlabel("strike K")
ax2.set_ylabel("σ error (vol points × 100)")
ax2.set_title("Repricing error: σ_imp(MC) − σ_imp(market)")
ax2.legend()
plt.tight_layout()
plt.show()

# Tabulate.
print(f"{'K':>6}  {'market':>8}  {'ConstantVol':>14}  {'BlackTS':>12}  {'Dupire':>10}")
for j, K in enumerate(strike_grid):
    print(
        f"{K:6.1f}  {baseline[j]:8.4f}"
        f"  {prices_const[j]:8.4f} (±{se_const[j]:.4f})"
        f"  {prices_term[j]:6.4f} (±{se_term[j]:.4f})"
        f"  {prices_dup[j]:5.4f} (±{se_dup[j]:.4f})"
    )

## Summary

- **Constant vol** prices a flat smile — every strike comes out at the same implied vol. On an equity surface with negative skew the OTM puts are massively under-priced and the OTM calls are over-priced (or vice versa, depending on the chosen $\sigma$).
- **`BlackTermStructureVol`** matches the ATM column at every expiry by construction (it inverts the relation $\theta_T = \int_0^T \sigma_\text{inst}^2 du$). At the ATM strike it agrees with the market within MC noise; away from ATM it still produces a flat smile, just at the right ATM level.
- **`DupireLocalVol`** reprices the **full** strip — every $(T, K)$ comes back within ~1 MC standard error, *including the skew wings*. That is the whole point: Dupire is the unique Markov-in-$S$ diffusion whose vanilla prices match the input surface by construction.

The MC repricing here is approximate (50 k paths, plain Euler, no variance reduction). The deterministic anchor in PR 4 is the closed-form Dupire formula plus identity checks (flat-vol gives $\sigma_\text{loc} = \sigma_\text{imp}$; pure-time-dependent agrees with `BlackTermStructureVol` to machine precision; analytical SVI/SSVI matches central-FD on the same surface). When the proper path engine arrives — antithetic, Brownian-bridge dimension allocation, Sobol QMC — the same notebook (or its successor test) will tighten the agreement to a fraction of a vol point at every strike.

See `tasks/todo.md` for the deferred MC repricing test that picks this up at production rigour.